In [1]:
!pip install transformers datasets scikit-learn torch

In [2]:
import pandas as pd
import numpy as np
import torch
import random
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import precision_recall_fscore_support

SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
train_df = pd.read_csv("/kaggle/input/datasets/spartacus00/smm4h-heard-2026-task5-test/SMM4H-HeaRD-2026-Task5-Training.tsv", sep="\t")
valid_df = pd.read_csv("/kaggle/input/datasets/spartacus00/smm4h-heard-2026-task5-test/SMM4H-HeaRD-2026-Task5-Validation.tsv", sep="\t")
test_df = pd.read_csv("/kaggle/input/datasets/spartacus00/smm4h-heard-2026-task5-test/SMM4H-HeaRD-2026-Task5-Test-Unlabeled.tsv", sep="\t")
# print(train_df.head())
train_all_df = pd.concat([train_df, valid_df], ignore_index=True)
print(train_all_df.shape)


(17718, 3)


In [4]:
class PubMedDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=256):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = str(self.data.iloc[idx]["text"])
        label = int(self.data.iloc[idx]["label"])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [5]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    return {"precision": precision, "recall": recall, "f1": f1}

In [7]:
# ── Model 1: BiomedBERT (base, abstract + fulltext) ──────────────────────────
model_name_a = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

tokenizer_a = AutoTokenizer.from_pretrained(model_name_a)

# Train on train + validation
train_dataset_a = PubMedDataset(train_all_df, tokenizer_a)
valid_dataset_a = PubMedDataset(valid_df, tokenizer_a)

model_a = AutoModelForSequenceClassification.from_pretrained(model_name_a, num_labels=2)

training_args_a = TrainingArguments(
    output_dir="./results_BiomedBERT",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    save_only_model=True,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    fp16=True,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=43
)

trainer_a = Trainer(
    model=model_a,
    args=training_args_a,
    train_dataset=train_dataset_a,
    eval_dataset=valid_dataset_a,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer_a.train()
print("\nBiomedBERT validation metrics:")
trainer_a.evaluate(valid_dataset_a)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,1.796154,0.241359,0.898785,0.755102,0.820702
2,1.066588,0.125522,0.867692,0.959184,0.911147
3,0.797352,0.074804,0.911672,0.982993,0.945990
4,0.544912,0.048164,0.948387,1.000000,0.973510
5,0.459775,0.028952,0.986532,0.996599,0.991540
6,0.286755,0.018930,0.989899,1.000000,0.994924
7,0.218962,0.019450,0.986532,0.996599,0.991540
8,0.123126,0.014560,0.989899,1.000000,0.994924


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


BiomedBERT validation metrics:


{'eval_loss': 0.018949151039123535,
 'eval_precision': 0.98989898989899,
 'eval_recall': 1.0,
 'eval_f1': 0.9949238578680203,
 'eval_runtime': 27.5475,
 'eval_samples_per_second': 80.37,
 'eval_steps_per_second': 10.055,
 'epoch': 8.0}

In [ ]:
# ── Model 2: PubMedBERT (base, abstracts only) ────────────────────────────────
model_name_b = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

tokenizer_b = AutoTokenizer.from_pretrained(model_name_b)

# Train on train + validation
train_dataset_b = PubMedDataset(train_all_df, tokenizer_b)
valid_dataset_b = PubMedDataset(valid_df, tokenizer_b)

model_b = AutoModelForSequenceClassification.from_pretrained(model_name_b, num_labels=2)

training_args_b = TrainingArguments(
    output_dir="./results_PubMedBERT",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    save_only_model=True,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    fp16=False,
    weight_decay=0.01,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    seed=42
)

trainer_b = Trainer(
    model=model_b,
    args=training_args_b,
    train_dataset=train_dataset_b,
    eval_dataset=valid_dataset_b,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer_b.train()
print("\nPubMedBERT validation metrics:")
trainer_b.evaluate(valid_dataset_b)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


PubMedBERT validation metrics:


{'eval_loss': 0.012028059922158718,
 'eval_precision': 0.9932432432432432,
 'eval_recall': 1.0,
 'eval_f1': 0.9966101694915255,
 'eval_runtime': 27.3062,
 'eval_samples_per_second': 81.08,
 'eval_steps_per_second': 10.144,
 'epoch': 9.0}

In [10]:
class PubMedDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, has_labels=True):
        self.data = data
        self.tokenizer = tokenizer
        self.has_labels = has_labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = str(self.data.iloc[idx]["text"])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.has_labels and "label" in self.data.columns:
            item["labels"] = int(self.data.iloc[idx]["label"])
        return item

In [11]:
# ── Predict# ── Predict on test set ────────────────────────────────────────────────
test_dataset_a = PubMedDataset(test_df, tokenizer_a, has_labels=False)
test_dataset_b = PubMedDataset(test_df, tokenizer_b, has_labels=False)

preds_a = trainer_a.predict(test_dataset_a)
preds_b = trainer_b.predict(test_dataset_b)

logits_a = preds_a.predictions   # shape: (N, 2)
logits_b = preds_b.predictions   # shape: (N, 2)

print(f"Logits shape — Model A: {logits_a.shape}, Model B: {logits_b.shape}")

Logits shape — Model A: (4429, 2), Model B: (4429, 2)


In [12]:
# ── Soft-voting ensemble (average softmax probabilities) ──────────────
def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))   # numerically stable
    return e / e.sum(axis=1, keepdims=True)

probs_a = softmax(logits_a)   # (N, 2)
probs_b = softmax(logits_b)   # (N, 2)

# Equal-weight average (adjust weights if needed)
avg_probs = 0.5 * probs_a + 0.5 * probs_b

# Final ensemble predictions with threshold
THRESHOLD = 0.46
ensemble_preds = (avg_probs[:, 1] >= THRESHOLD).astype(int)


In [13]:
# ── Save ensemble predictions for submission ─────────────────────────
pmcids = test_df["pmcid"].tolist()

df_out = pd.DataFrame({
    "pmcid": pmcids,
    "label": ensemble_preds
})

df_out.to_csv("test_predictions_Ensemble.tsv", sep="\t", index=False)
print("Saved predictions to test_predictions_Ensemble.tsv")
print(f"Label distribution:\n{df_out['label'].value_counts()}")

Saved predictions to test_predictions_Ensemble.tsv
Label distribution:
label
0    3777
1     652
Name: count, dtype: int64
